# Retrieval Stage: Surprise SVD

This notebook:
1. Loads cleaned ratings data
2. Trains a Surprise SVD model
3. Implements candidate retrieval function
4. Saves the trained model


In [11]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from surprise import SVD, Dataset, Reader
from collections import defaultdict

# Set up paths
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

CONFIG = {
    "svd": {
        "n_factors": 64,
        "n_epochs": 30,
        "lr_all": 0.005,
        "reg_all": 0.02,
        "random_state": 42,
    },
    "topN": 200,
}

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed directory: {PROCESSED_DIR}")
print(f"Models directory: {MODELS_DIR}")
print(f"SVD config: {CONFIG['svd']}")



Project root: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system
Processed directory: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/data/processed
Models directory: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/models
SVD config: {'n_factors': 64, 'n_epochs': 30, 'lr_all': 0.005, 'reg_all': 0.02, 'random_state': 42}


In [12]:
# Load cleaned ratings
ratings_path = PROCESSED_DIR / "ratings_clean.parquet"
df_ratings = pd.read_parquet(ratings_path)

# Ensure timestamp exists
assert 'timestamp' in df_ratings.columns, "ratings_clean must include 'timestamp' for time-based split"

# Time-based split (sorted by timestamp)
df_ratings = df_ratings.sort_values('timestamp').reset_index(drop=True)
split_idx = int(len(df_ratings) * 0.8)
train_df = df_ratings.iloc[:split_idx].copy()
holdout_df = df_ratings.iloc[split_idx:].copy()

print(f"Full ratings shape: {df_ratings.shape}")
print(f"Train shape: {train_df.shape}, Holdout shape: {holdout_df.shape}")
print(f"Unique users (train): {train_df['user_id'].nunique()}")
print(f"Unique items (train): {train_df['parent_asin'].nunique()}")


Full ratings shape: (701528, 4)
Train shape: (561222, 4), Holdout shape: (140306, 4)
Unique users (train): 510379
Unique items (train): 92369


In [13]:
def train_mf(train_df: pd.DataFrame):
    """Train Surprise SVD on train_df and return model and trainset."""
    reader = Reader(rating_scale=(1, 5))
    data = Dataset.load_from_df(train_df[['user_id', 'parent_asin', 'rating']], reader)
    trainset = data.build_full_trainset()
    model = SVD(**CONFIG['svd'])
    model.fit(trainset)
    return model, trainset

def predict_mf(model, trainset, user_id: str, item_id: str) -> float:
    """Predict rating; return global mean if user/item unseen."""
    try:
        uid = trainset.to_inner_uid(user_id)
        iid = trainset.to_inner_iid(item_id)
        return model.predict(uid, iid).est
    except ValueError:
        return trainset.global_mean

def retrieve_candidates(model, trainset, user_id: str, items_pool: list, user_seen: dict, topN: int = 200):
    """Retrieve topN items from items_pool excluding seen items."""
    seen = user_seen.get(user_id, set())
    candidates = []
    for asin in items_pool:
        if asin in seen:
            continue
        try:
            iid = trainset.to_inner_iid(asin)
            uid = trainset.to_inner_uid(user_id)
            est = model.predict(uid, iid).est
            candidates.append((asin, est))
        except ValueError:
            continue
    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:topN]

print("Training MF model on train split...")
mf_model, mf_trainset = train_mf(train_df)
print("MF training complete!")
print(f"Trainset users: {mf_trainset.n_users}, items: {mf_trainset.n_items}, ratings: {mf_trainset.n_ratings}")


Training MF model on train split...
MF training complete!
Trainset users: 510379, items: 92369, ratings: 561222


In [14]:
# Precompute user->seen mapping from train split
user_seen_items = defaultdict(set)
for _, row in train_df.iterrows():
    user_seen_items[row['user_id']].add(row['parent_asin'])

items_pool = train_df['parent_asin'].unique().tolist()

print(f"Items pool size (train items): {len(items_pool)}")
print(f"Users with interactions (train): {len(user_seen_items)}")

Items pool size (train items): 92369
Users with interactions (train): 510379


In [15]:
# Quick test of retrieval
test_user = train_df['user_id'].iloc[0]
top_candidates = retrieve_candidates(
    mf_model,
    mf_trainset,
    user_id=test_user,
    items_pool=items_pool,
    user_seen=user_seen_items,
    topN=20,
)

print(f"Test user: {test_user}")
print(f"Retrieved {len(top_candidates)} candidates; top 5 shown:")
for asin, est in top_candidates[:5]:
    print(f"  {asin}: {est:.4f}")

Test user: AED2GFGIAJ22PHMZGSKH2CPUF75Q
Retrieved 20 candidates; top 5 shown:
  B000050AUD: 4.0150
  B000050FDB: 4.0150
  B00005JKQ4: 4.0150
  B000050B65: 4.0150
  B000068PBR: 4.0150


In [16]:
# Save model and helper artifacts
model_path = MODELS_DIR / "svd_surprise.pkl"

artifact = {
    "model": mf_model,
    "trainset": mf_trainset,
    "items_pool": items_pool,
    "user_seen_items": dict(user_seen_items),
    "config": CONFIG,
}

with open(model_path, "wb") as f:
    pickle.dump(artifact, f)

print(f"Saved MF model and artifacts to {model_path}")
print(f"File size: {model_path.stat().st_size / (1024*1024):.2f} MB")


Saved MF model and artifacts to /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/models/svd_surprise.pkl
File size: 345.76 MB


In [17]:
# Save model and helper data
model_path = MODELS_DIR / "svd_surprise.pkl"

# Create a dictionary with model and helper data
model_data = {
    'model': model,
    'trainset': trainset,
    'user_seen_items': dict(user_seen_items),  # Convert defaultdict to dict
    'all_items': list(all_items)  # Convert set to list for pickle
}

with open(model_path, 'wb') as f:
    pickle.dump(model_data, f)

print(f"Saved model and helper data to: {model_path}")
print(f"Model file size: {model_path.stat().st_size / (1024*1024):.2f} MB")


Saved model and helper data to: /Users/charukagunawardhane/Documents/Development/ML_DL/LocalEnv-recommendation-system/models/svd_surprise.pkl
Model file size: 360.95 MB
